In [ ]:
!pip install langchain faiss-cpu transformers sentence-transformers pymupdf -q
!pip install -U langchain-community -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 94.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

In [1]:
import os
from google.colab import drive

drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

##Indexing

##Law Books Splitting

In [ ]:
!pip install PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 15.1 MB/s eta 0:00:00


In [ ]:
import os
from PyPDF2 import PdfReader
import re
from langchain.schema import Document
import json


#PDF Path

folder_path = "/content/drive/MyDrive/LegalMind"
lawbook_paths = [
    f"{folder_path}/Code Of Criminal Procedure 1898.pdf",
    f"{folder_path}/Pakistan Penal Code.pdf",
    f"{folder_path}/Qanun-e-Shahadat Order 1984.pdf"
]


#Split by Numbered Sections

def split_into_documents(text, book_name):
    lines = text.split('\n')
    documents = []
    current_chunk = ""
    section_number = None

    section_header_pattern = re.compile(r'^\s*(\d+)\.\s')

    for line in lines:
        match = section_header_pattern.match(line)
        if match:
            if current_chunk:
                doc = Document(
                    page_content=current_chunk.strip(),
                    metadata={"book": book_name, "section": section_number}
                )
                documents.append(doc)
                current_chunk = ""
            section_number = match.group(1)
            current_chunk = line
        else:
            current_chunk += '\n' + line


#Save last chunk

    if current_chunk:
        doc = Document(
            page_content=current_chunk.strip(),
            metadata={"book": book_name, "section": section_number}
        )
        documents.append(doc)

    return documents


#Process All Books

all_chunks = []

for path in lawbook_paths:
    book_name = os.path.basename(path).replace(".pdf", "")
    print(f"Processing: {book_name}")

    reader = PdfReader(path)
    text = "\n".join(page.extract_text() or "" for page in reader.pages)

    docs = split_into_documents(text, book_name)
    all_chunks.extend(docs)

    print(f"{len(docs)} chunks created for {book_name}\n")

#Save lawbook_chunks to a JSON file
lawbook_raw_data = [{"content": doc.page_content, "metadata": doc.metadata} for doc in all_chunks]
with open("lawbook_chunks.json", "w") as f:
    json.dump(lawbook_raw_data, f)

Processing: Code Of Criminal Procedure 1898
568 chunks created for Code Of Criminal Procedure 1898

Processing: Pakistan Penal Code
517 chunks created for Pakistan Penal Code

Processing: Qanun-e-Shahadat Order 1984
260 chunks created for Qanun-e-Shahadat Order 1984



In [ ]:
import random

sample_chunks = random.sample(all_chunks, 10)

for i, chunk in enumerate(sample_chunks):
    print(f"\nSample Chunk #{i+1}")
    print("-" * 50)
    print(f"Book: {chunk.metadata['book']}")
    print(f"Section: {chunk.metadata['section']}")
    print("Content Preview:")
    print(chunk.page_content, "...\n")


Sample Chunk #1
--------------------------------------------------
Book: Code Of Criminal Procedure 1898
Section: 389
Content Preview:
389. Who may issue warrant.  390. Execution of sentence of whipping only.  391. Execution of sentence of whippi ng, in addition to imprisonment. ...


Sample Chunk #2
--------------------------------------------------
Book: Qanun-e-Shahadat Order 1984
Section: 96
Content Preview:
96. presumption as to certified copies of  foreign Judicial records                        ...    39 ...


Sample Chunk #3
--------------------------------------------------
Book: Code Of Criminal Procedure 1898
Section: 561
Content Preview:
561. [repealed].  561 -A. Saving of inherent  power of High Court. ...


Sample Chunk #4
--------------------------------------------------
Book: Qanun-e-Shahadat Order 1984
Section: 105
Content Preview:
105. Evidence as to document unmeaning in reference to existing facts:  When 
language used in a document is plain in itsel f, but is unm

##Judgments Splitting

In [ ]:
import re
import json
from langchain.document_loaders import PyMuPDFLoader
from langchain.schema import Document


def clean_text(text):
    text = text.lower()
    text = text.replace("_", "").replace("-", "")
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return "\n".join(lines)

def is_new_judgment_page(lines):
    if not lines:
        return False

    lines = [line.lower().strip() for line in lines[:12]]
    joined = " ".join(lines)

    return any([
        re.search(r"judg[e]?ment\s+sheet", joined),
        re.search(r"peshawar high court", joined),
        re.search(r"\b(cr\.a|crl\.a|wp no|w\.p\.|writ petition|cr\.r|cr\.revision|c\.p\.|c\.a\.|c\.r\.|civil appeal|civil revision|criminal appeal|criminal misc|crl\.misc|misc\. appl)\b", joined),
        re.match(r"^[a-z]{2,4}\.\s?no\.\s?\d{1,4}", lines[0]) if lines else False
    ])

def extract_court_name(text):
    first_3_lines = "\n".join(text.strip().split("\n")[:3]).lower()
    if "lahore high court" in first_3_lines or "lahore" in first_3_lines:
        return "Lahore High Court"
    elif "peshawar high court" in first_3_lines or "peshawar" in first_3_lines:
        return "Peshawar High Court"
    else:
        return "Unknown"

def split_judgments_by_common_headers(pdf_path):
    loader = PyMuPDFLoader(pdf_path)
    pages = loader.load()

    judgments = []
    current = []

    for page in pages:
        lines = page.page_content.strip().splitlines()
        if is_new_judgment_page(lines) and current:
            combined_text = "\n\n".join([p.page_content for p in current])
            cleaned_text = clean_text(combined_text)
            court = extract_court_name(cleaned_text)

            judgments.append(Document(
                page_content=cleaned_text,
                metadata={
                    "source": f"Judgment {len(judgments) + 1}",
                    "court": court
                }
            ))
            current = [page]
        else:
            current.append(page)

    if current:
        combined_text = "\n\n".join([p.page_content for p in current])
        cleaned_text = clean_text(combined_text)
        court = extract_court_name(cleaned_text)

        judgments.append(Document(
            page_content=cleaned_text,
            metadata={
                "source": f"Judgment {len(judgments) + 1}",
                "court": court
            }
        ))

    return judgments


#Save lawbook to a JSON file

pdf_path = "/content/drive/MyDrive/LegalMind/Judgments.pdf"

judgments = split_judgments_by_common_headers(pdf_path)

judgment_raw_data = [{"content": doc.page_content, "metadata": doc.metadata} for doc in judgments]
with open("judgments_chunks.json", "w", encoding="utf-8") as f:
    json.dump(judgment_raw_data, f)


In [ ]:
from collections import Counter

total_judgments = len(judgments)


court_counts = Counter(doc.metadata.get("court", "Unknown") for doc in judgments)

print(f"Total judgments: {total_judgments}\n")
print("Judgments per court:")
for court, count in court_counts.items():
    print(f"{court}: {count}")


Total judgments: 1010

Judgments per court:
Peshawar High Court: 368
Unknown: 571
Lahore High Court: 71


In [ ]:
for i, doc in enumerate(judgments[:3], 1):
    print(f"\nJudgment #{i}")
    print("-" * 50)
    print(f"Source: {doc.metadata.get('source', 'Unknown')}")
    print(f"Court: {doc.metadata.get('court', 'Unknown')}")
    print("Full Content:")
    print(doc.page_content)
    print("\n" + "="*50)



Judgment #1
--------------------------------------------------
Source: Judgment 1
Court: Peshawar High Court
Full Content:
judement sheet
peshawar high court, abbottabad bench.
judicial department
cr.a no.323a of 2019
judga,ient
date of hearing
... ...21.04.2020
appellant...(the state) by raja muhammad zubair, additional advocate
general ....
respondent ... (mir umar son of sain muhammad)
v
ahmad ali, j:
the state through
advocate general, khyber pakhtunkhwa has called in
question the acquittal of accused/respondent, in case
f.i.r no. 182 dated \7.04.2015 under section 302
ppc of ps shinkiari, district mansehra, vide
impugned judgment dated 11.03.2019 of the learned
additional sessions judgeiii, mansehra by filing
instant appeal under section 417 (2a) cr.p.c.
2.
brief facts of the case are that the
complainant, mst. bibi iratima wife of said khan,
reported the matter to thc local policc on 17 '042015
at 15.30 hours, u,ho reached the spot in village khara
battangi cum makra miana that 

##Upload Chunks

In [ ]:
from google.colab import files

uploaded = files.upload()

for filename in uploaded.keys():
    print(f"Uploaded file: {filename}")


Saving judgment_store.zip to judgment_store (1).zip
Saving lawbook_store.zip to lawbook_store (1).zip
Uploaded file: judgment_store (1).zip
Uploaded file: lawbook_store (1).zip


##InLegalBERTEmbedder: Custom Mean-Pooling Embedder for Legal Texts

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F

class InLegalBERTEmbedder:
    def __init__(self, model_name="law-ai/InLegalBERT", device=None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(self.device)

    def embed_documents(self, texts):
        embeddings = []
        for text in texts:
            inputs = self.tokenizer(text, padding=True, truncation=True, return_tensors="pt", max_length=512).to(self.device)
            with torch.no_grad():
                model_output = self.model(**inputs)

#Mean Pooling

            token_embeddings = model_output.last_hidden_state
            attention_mask = inputs["attention_mask"]
            input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
            sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
            sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
            embedding = sum_embeddings / sum_mask
            embeddings.append(embedding.squeeze(0).cpu().numpy())
        return embeddings

In [ ]:
embedder = InLegalBERTEmbedder()
vectors = embedder.embed_documents(["This is a legal sentence."])
vectors

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/516 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/671 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/534M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/534M [00:00<?, ?B/s]

[array([ 2.21118420e-01,  6.73190504e-02,  3.82074803e-01,  4.00571153e-02,
         2.42574349e-01, -2.41989151e-01,  3.99464853e-02,  1.76098511e-01,
        -1.96857631e-01, -1.89602792e-01,  3.30429643e-01, -1.59139648e-01,
         1.65412545e-01, -2.30779439e-01, -1.24097452e-01,  5.12865186e-02,
         9.52035189e-04, -2.26209432e-01,  4.54799026e-01, -1.76061720e-01,
         2.48421714e-01,  2.43738592e-01, -3.71059954e-01,  3.80338728e-02,
         1.93377137e-01,  1.77119136e-01,  4.46843281e-02, -1.06362604e-01,
        -1.28440112e-01, -9.17349011e-02,  2.76923239e-01,  1.96417287e-01,
         1.38730526e-01,  1.58120006e-01, -1.47234082e-01, -8.63551646e-02,
         5.45994639e-02, -2.89401382e-01,  1.11716710e-01, -1.53467029e-01,
        -1.75575987e-01,  1.79016888e-01,  3.04955423e-01, -1.74870238e-01,
        -2.30879024e-01,  4.10582423e-02, -2.92121470e-01, -3.39809060e-01,
        -2.17263401e-02, -5.03929496e-01, -3.90288755e-02, -3.72597799e-02,
        -1.8

In [ ]:
print(type(vectors))
print(vectors[0].shape)

<class 'list'>
(768,)


## LangChain Wrapper for InLegalBERT Embeddings

In [ ]:
from langchain.embeddings.base import Embeddings

class InLegalBERTLangChain(Embeddings):
    def __init__(self):
        self.model = InLegalBERTEmbedder()

    def embed_documents(self, texts):
        return self.model.embed_documents(texts)

    def embed_query(self, text):
        return self.model.embed_documents([text])[0]

##Vector Stores (FAISS)

In [ ]:
from langchain.vectorstores import FAISS

def create_vectorstores(judgment_docs, lawbook_chunks):
    embeddings = InLegalBERTLangChain()


#Lawbook Store

    lawbook_store = FAISS.from_documents(lawbook_chunks, embeddings)
    lawbook_store.save_local("/content/lawbook_store")
    print("Lawbook vector store saved.")


# Judgment Store

    judgment_store = FAISS.from_documents(judgment_docs, embeddings)
    judgment_store.save_local("/content/judgment_store")
    print("Judgment vector store saved.")

    return lawbook_store, judgment_store, embeddings

In [ ]:
judgment_store, lawbook_store, embeddings = create_vectorstores(judgments, all_chunks)

tokenizer_config.json:   0%|          | 0.00/516 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/671 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/534M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/534M [00:00<?, ?B/s]

Lawbook vector store saved.
Judgment vector store saved.


##Upload Vector Stores

In [ ]:
from google.colab import files
uploaded = files.upload()

import zipfile
import os


with zipfile.ZipFile("judgment_store.zip", 'r') as zip_ref:
    zip_ref.extractall("/content/judgment_store")


with zipfile.ZipFile("lawbook_store.zip", 'r') as zip_ref:
    zip_ref.extractall("/content/lawbook_store")


Saving judgment_store.zip to judgment_store.zip
Saving lawbook_store.zip to lawbook_store.zip


In [ ]:
!pip install -U langchain-community

In [ ]:
embeddings = InLegalBERTLangChain()

from langchain.vectorstores import FAISS

lawbook_store = FAISS.load_local("/content/lawbook_store", embeddings, allow_dangerous_deserialization=True)
judgment_store = FAISS.load_local("/content/judgment_store", embeddings, allow_dangerous_deserialization=True)


##OpenAI LLM Initialization

In [ ]:
!pip install -U langchain-openai

In [ ]:
from langchain_openai import ChatOpenAI
import os
os.environ["OPENAI_API_KEY"] = "API Key"


llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)

##RAG Law Book

##Retrieval

In [ ]:
retriever_law = lawbook_store.as_retriever(search_type="similarity", search_kwargs={"k": 3})

In [ ]:
retriever_law

VectorStoreRetriever(tags=['FAISS', 'InLegalBERTLangChain'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7cc285999250>, search_kwargs={'k': 3})

In [ ]:
sections = retriever_law.invoke('Explain the offence defined in Section 375 PPC and its punishment.')
sections

[Document(id='e1f832bd-982c-46d6-9112-3c38ebd5e0c9', metadata={'book': 'Code Of Criminal Procedure 1898', 'section': '82'}, page_content='82. Where warrant may be executed:  A warrant of arrest may be executed at any place \nin Pakistan.  \n [Explanation : In this section,  "warrant of arrest" includes a warrant of arrest issued under \nthis Code as in force in Azad Jammu and Kashmir]  \nExplan. added by Code of Criminal Procedure (Amendment) Act. Vlll of 1993.'),
 Document(id='6b5fe735-3d7f-46ef-8013-e2adca6b3f15', metadata={'book': 'Pakistan Penal Code', 'section': '101'}, page_content='101. When such right extends to causing any harm other than death:  \nIf the offence be not of any of the descriptions  enumerated in the last  preceding section, the \nright of private defence of th e body dose not extend to the voluntary causing of death to the \nassailant, but dose extend, under the restrictions mentioned in Section 99 to the voluntary \ncausing to the assailant of a ny harm other 

In [ ]:
sections[0].page_content

'82. Where warrant may be executed:  A warrant of arrest may be executed at any place \nin Pakistan.  \n [Explanation : In this section,  "warrant of arrest" includes a warrant of arrest issued under \nthis Code as in force in Azad Jammu and Kashmir]  \nExplan. added by Code of Criminal Procedure (Amendment) Act. Vlll of 1993.'

##Augmentation

In [ ]:
from langchain.prompts import PromptTemplate

prompt_law = PromptTemplate(
    input_variables=["context", "query"],
    template="""
You are a highly competent legal assistant specializing in Pakistani criminal law.

Use the following sections from Pakistan law books to answer the legal query.

For each section, write a separate explanation — even if the section is only partially relevant. Use this format:

---

Section [Section Number] – [Title or Short Description]
From: [Book Name]

[Explain how this section is relevant to the query.]

---

If a section is completely irrelevant, ignore it entirely and do not mention it in the answer.

Respond in formal legal language only.

Context:
{context}

Query:
{query}
"""
)

In [ ]:
query = "What is the procedure for issuing a warrant of arrest in Pakistan?"
retrieved_sections = retriever_law.invoke(query)
retrieved_sections

[Document(id='e1f832bd-982c-46d6-9112-3c38ebd5e0c9', metadata={'book': 'Code Of Criminal Procedure 1898', 'section': '82'}, page_content='82. Where warrant may be executed:  A warrant of arrest may be executed at any place \nin Pakistan.  \n [Explanation : In this section,  "warrant of arrest" includes a warrant of arrest issued under \nthis Code as in force in Azad Jammu and Kashmir]  \nExplan. added by Code of Criminal Procedure (Amendment) Act. Vlll of 1993.'),
 Document(id='64eb9069-6b22-4261-af00-2f3caaaf8769', metadata={'book': 'Code Of Criminal Procedure 1898', 'section': '186'}, page_content="186. Power to issue summons or warrant for offence committed beyond local jurisdiction. \nMagistrate's procedure on arrest."),
 Document(id='7765be99-e579-4a10-83f6-46408e38016e', metadata={'book': 'Code Of Criminal Procedure 1898', 'section': '85'}, page_content='85.   Procedure on arrest of per son against whom warrant issued.')]

In [ ]:
if not isinstance(retrieved_sections, list):
    retrieved_sections = [retrieved_sections]

context_text = "\n\n---\n\n".join([
    f"Section from {doc.metadata.get('book', 'Unknown Book')} (Section {doc.metadata.get('section', 'N/A')}):\n{doc.page_content}"
    for doc in retrieved_sections
])

In [ ]:
final_prompt_law = prompt_law.invoke({"context": context_text, "query": query})
final_prompt_law

StringPromptValue(text='\nYou are a highly competent legal assistant specializing in Pakistani criminal law.\n\nUse the following sections from Pakistan law books to answer the legal query.\n\nFor each section, write a separate explanation — even if the section is only partially relevant. Use this format:\n\n---\n\nSection [Section Number] – [Title or Short Description]\nFrom: [Book Name]\n\n[Explain how this section is relevant to the query.]\n\n---\n\nIf a section is completely irrelevant, ignore it entirely and do not mention it in the answer.\n\nRespond in formal legal language only.\n\nContext:\nSection from Code Of Criminal Procedure 1898 (Section 82):\n82. Where warrant may be executed:  A warrant of arrest may be executed at any place \nin Pakistan.  \n [Explanation : In this section,  "warrant of arrest" includes a warrant of arrest issued under \nthis Code as in force in Azad Jammu and Kashmir]  \nExplan. added by Code of Criminal Procedure (Amendment) Act. Vlll of 1993.\n\n-

##Generation

In [ ]:
answer = llm.invoke(final_prompt_law)
print(answer.content)

---

Section 82 – Where warrant may be executed
From: Code Of Criminal Procedure 1898

Section 82 of the Code of Criminal Procedure 1898 states that a warrant of arrest may be executed at any place in Pakistan. This means that the warrant can be enforced throughout the country, regardless of where the alleged offender is located. It is important to note that this section also includes warrants of arrest issued under the Code as in force in Azad Jammu and Kashmir.

---

Section 85 – Procedure on arrest of person against whom warrant issued
From: Code Of Criminal Procedure 1898

Section 85 of the Code of Criminal Procedure 1898 outlines the procedure to be followed upon the arrest of a person against whom a warrant has been issued. This section details the steps that must be taken by the arresting officer, including informing the arrested individual of the grounds of their arrest and producing them before the court as soon as possible. It also covers the circumstances under which the arr

##Chain

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

def format_docs(docs):
    return "\n\n---\n\n".join([
        f"Section {doc.metadata.get('section', 'N/A')} – {doc.metadata.get('title', 'No Title')}\n\n"
        f"From: {doc.metadata.get('book', 'Unknown Book')}\n\n"
        f"{doc.page_content.strip()}"
        for doc in docs
    ])

parallel_chain_law = RunnableParallel({
    "context": retriever_law | RunnableLambda(format_docs),
    "query": RunnablePassthrough()
})

parser = StrOutputParser()

main_chain_law = parallel_chain_law | prompt_law | llm | parser

In [ ]:
response = main_chain_law.invoke("What are the legal procedures for search and seizure during a criminal investigation under Pakistani law?")
print(response)

---

Section 5 – Trial of offences under Penal Code
From: Code Of Criminal Procedure 1898

This section outlines that all offences under the Pakistan Penal Code shall be investigated, enquired into, tried, and otherwise dealt with according to the provisions contained within the Code of Criminal Procedure. This means that the legal procedures for search and seizure during a criminal investigation under Pakistani law would be governed by the provisions outlined in the Code of Criminal Procedure for offences under the Penal Code.

---

Section 58 – Pursuit of offenders into other jurisdiction
From: Code Of Criminal Procedure 1898

This section allows a police officer to pursue and arrest, without a warrant, any person authorized to be arrested under the Code of Criminal Procedure in any place in Pakistan. This provision grants the authority to law enforcement to conduct searches and seizures during a criminal investigation by pursuing offenders into different jurisdictions within Pakista

##RAG Judgment

##Retriever

In [ ]:
retriever_judgment = judgment_store.as_retriever(search_type="similarity", search_kwargs={"k": 3})
retriever_judgment

VectorStoreRetriever(tags=['FAISS', 'InLegalBERTLangChain'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7cc28732dc10>, search_kwargs={'k': 3})

In [ ]:
prompt_judgment = retriever_judgment.invoke('Explain the offence defined in Section 375 PPC and its punishment.')
prompt_judgment

[Document(id='e078e5a1-84c6-4999-a503-9fa7f6e3aa1f', metadata={'source': 'Judgment 588', 'court': 'Unknown'}, page_content="writ petition no.9549/2021\n(4)\nrelating to the commission of a cognizable offence and to\nregister a case thereon on the ground that he is not satisfied\nwith the reasonableness or credibility of the information. in\nother words, 'reasonableness' or 'credibility' of the said\ninformation is not a condition precedent for the registration of\na criminal case. a comparison of the present section 154 of\nthe code with those of the earlier codes indicates that the\nlegislature had intentionally thought it appropriate to employ\nonly the words 'every information' without qualifying the said\nwords. an overall reading of all the codes makes it clear that\nsine qua non for recording a first information report is that\nthere should be an information and that information must\ndisclose the commission of a cognizable offence.\n6.\nsection 156 of the code confers the power 

In [ ]:
prompt_judgment[0].page_content

"writ petition no.9549/2021\n(4)\nrelating to the commission of a cognizable offence and to\nregister a case thereon on the ground that he is not satisfied\nwith the reasonableness or credibility of the information. in\nother words, 'reasonableness' or 'credibility' of the said\ninformation is not a condition precedent for the registration of\na criminal case. a comparison of the present section 154 of\nthe code with those of the earlier codes indicates that the\nlegislature had intentionally thought it appropriate to employ\nonly the words 'every information' without qualifying the said\nwords. an overall reading of all the codes makes it clear that\nsine qua non for recording a first information report is that\nthere should be an information and that information must\ndisclose the commission of a cognizable offence.\n6.\nsection 156 of the code confers the power upon a\npolice officer to investigate a cognizable offence whereas\nsection 157 lays down the manner, in which that investi

##Augmentation

In [ ]:
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt_judgment = ChatPromptTemplate.from_template("""
You are a legal assistant trained in Pakistani law. Given the following legal judgment text, extract and summarize the key legal information in a structured manner.

Format the output like this:

Case Type: [e.g. Writ Petition, Criminal Appeal]
Court: [e.g. Lahore High Court, Supreme Court of Pakistan]
Parties Involved:
- Petitioner(s): [Names if available]
- Respondent(s): [Names if available]

Main Legal Issues:
[Summarize the core legal issues raised in the case.]

Petitioner’s Arguments:
[Summarize what the petitioner argued.]

Respondent’s Arguments:
[Summarize what the respondent argued.]

Relevant Laws and Sections:
[List all mentioned laws/sections (e.g., Section 154 CrPC, Article 10A of the Constitution).]

Court’s Reasoning and Observations:
[Summarize how the court analyzed the matter, referring to any case law, principles, or interpretations.]

Final Decision / Order:
[Summarize what the court ordered — dismissed, allowed, directions issued, etc.]

---
Summary of the Judgment:
[Provide a concise summary of the entire judgment in no more than 10 lines.]

Judgment Text:
{context}
""")


In [ ]:
query = "Summarize the judgment regarding registration of FIR under section 154 CrPC"
retrieved_judgments = retriever_judgment.invoke(query)
retrieved_judgments

[Document(id='97aefde0-066b-4975-ac8f-9deb467c08a0', metadata={'source': 'Judgment 234', 'court': 'Unknown'}, page_content='a\njudgment sheet\njvsiqiak. pspabtmpiit\'\ncr. m. b.a no. 1959p/2024\nfaisal hussain\nvs\nthe state\ndate of hearins: 16.08.2024\npetitioner(s) by: mr. aman ullah pirzada, advocate\nstate by: mr. hazrat said, dag\ncomplainant by: arbab shabbir ahmad, advocate.\njudgment\n**{€rf*\nijaz anwar.j.\nthrough instant bail\napplication, accused petitioner faisal hussain son\nof iftikhar hussain seeks his release on bail in\ncase fir no.67124 dated 04.03.2024 registered\nunder section 40914191420 ppc, at police\nstation fia/cbc, peshawar.\n2.\nit is pertinent to mention here that the\nfirst bail application no. 3285p12023 filed by the\npetitioner in fir no. 258 dated 04.07,2023\n2\nregistered under section 40814191420 ppc, at\npolice station gharbi, peshawar was dismissed\nby this court on merit vide order dated\n13.09.2023. subsequently, another fir (present\none) has be

##Chain

In [ ]:
from langchain_core.runnables import RunnableLambda

judgment_summary_chain = RunnableLambda(
    lambda query: [
        (prompt_judgment | llm | StrOutputParser()).invoke({"context": doc.page_content})
        for doc in retriever_judgment.invoke(query)
    ]
)

In [ ]:
query = "criteria for pre-arrest bail in Pakistan"

response = judgment_summary_chain.invoke(query)


print_output = "\n\n-------\n\n".join(response)
print(print_output)


Case Type: Pre-arrest Bail Petition
Court: Not specified

Parties Involved:
- Petitioner(s): Not specified
- Respondent(s): Not specified

Main Legal Issues:
Granting pre-arrest bail, confirmation of ad-interim bail, furnishing of fresh bail bonds.

Petitioner’s Arguments:
The petitioner sought pre-arrest bail and requested confirmation of ad-interim bail.

Respondent’s Arguments:
Not specified

Relevant Laws and Sections:
Not specified

Court’s Reasoning and Observations:
The court allowed the pre-arrest bail petition, confirmed the ad-interim bail, and required the petitioner to furnish fresh bail bonds. The court emphasized that its findings were tentative and would not prejudice the trial.

Final Decision / Order:
The pre-arrest bail petition was allowed, and the ad-interim bail was confirmed, subject to the petitioner furnishing fresh bail bonds.

Summary of the Judgment:
The court allowed the pre-arrest bail petition, confirmed the ad-interim bail, and required the petitioner to 

##Legal Case Verdict Prediction Pipeline

In [ ]:
pip install langchain transformers torch scikit-learn

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from sklearn.preprocessing import LabelEncoder
from langchain.chat_models import ChatOpenAI
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser

import torch
import re
import json
import pandas as pd
import os
from google.colab import drive


#Load Trained Model & Tokenizer

model_path = "/content/drive/MyDrive/LegalMind/best_epoch18_model"
model = AutoModelForSequenceClassification.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)
model.eval()
print("Model & tokenizer loaded successfully.")


#Load Label Encoder

with open("/content/drive/MyDrive/LegalMind/LegalMind.json", "r", encoding="utf-8") as f:
    df = json.load(f)
df = pd.DataFrame(df)

label_encoder = LabelEncoder()
label_encoder.fit(df["verdict"])
label_mapping = dict(zip(label_encoder.transform(label_encoder.classes_), label_encoder.classes_))


#Text Cleaning

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[\n\r\t]", " ", text)
    text = re.sub(r"[\"']", "", text)
    text = re.sub(r"[^a-z0-9 ,.\[\]()/\-:]", "", text)
    text = re.sub(r"\.{2,}", ".", text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[,/]+", ",", text)
    text = re.sub(r"(,\s*,)+", ",", text)
    text = re.sub(r"(,\s*$)|(^\s*,)", "", text)
    text = re.sub(r"\b(p\s*,\s*p\s*,\s*c)\b", "ppc", text)
    return text.strip()

#Format Input
def format_case(case):
    summary = clean_text(case["summary"])
    pet = clean_text(case["petitioner_argument"])
    resp = clean_text(case["respondent_argument"])
    case_type = clean_text(case.get("case_type", ""))
    sections = clean_text(", ".join(case.get("offence_sections", [])))
    return f"[SUMMARY] {summary} [PETITIONER] {pet} [RESPONDENT] {resp} [CASE TYPE] {case_type} [SECTIONS] {sections}"


#Predict Function

def predict_legal_verdict(case: dict) -> str:
    formatted_text = format_case(case)
    inputs = tokenizer(formatted_text, return_tensors="pt", truncation=True, padding=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
        predicted_label = torch.argmax(outputs.logits, dim=1).item()
    return label_mapping[predicted_label]


#LLM Setup

llm = ChatOpenAI(temperature=0)

prompt_extract_case = PromptTemplate.from_template("""
Extract the following fields from the legal case text:
- summary
- petitioner_argument
- respondent_argument
- offence_sections (list like ["489-F", "420"])
- case_type

Respond in JSON format.

Case Text:
{text}
""")

output_parser = JsonOutputParser()

extract_case_chain = prompt_extract_case | llm | output_parser

#Run on Input Text
def run_pipeline(raw_text: str):
    print("Extracting structured case data from LLM...")
    structured_case = extract_case_chain.invoke({"text": raw_text})
    print("Structured fields:", json.dumps(structured_case, indent=2))

    print("Predicting verdict using LegalBERT classifier...")
    verdict = predict_legal_verdict(structured_case)
    print("Predicted Verdict:", verdict)
    return verdict

Model & tokenizer loaded successfully.


/tmp/ipython-input-45-3135918865.py:74: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  llm = ChatOpenAI(temperature=0)


##Tools

In [ ]:
from langchain.tools import tool
from langchain.schema import StrOutputParser




@tool
def lawbook_tool(query: str) -> str:
    """
    Use this tool when the user is asking about specific Pakistani laws, legal provisions, or statutory sections.
    It retrieves and summarizes relevant sections from digitized Pakistani law books (e.g., PPC, CrPC, PECA).

    Example queries:
    - "Explain Section 489-F about cheque dishonor."
    - "What does Section 302 of the Pakistan Penal Code say?"
    - "Give me details of cybercrime law under PECA."
    - "Is there any section in CrPC related to bail?"

    The tool returns a summarized explanation of the most relevant sections from law books.
    """
    k = 3  # Number of top sections to summarize

    # Retrieve relevant law book content
    retrieved = retriever_law.invoke(query)

    if not retrieved:
        return "No relevant law sections found."

    #FIX: Ensure this dictionary is defined
    section_titles = {}

    # Get top-k results
    top_docs = retrieved[:k]

    # Format context with metadata
    formatted_context = "\n\n---\n\n".join([
        (
            f"Section {doc.metadata.get('section', 'N/A')}"
            + (
                f" – {doc.metadata.get('title')}"
                if doc.metadata.get('title')
                else (
                    f" – {section_titles.get(doc.metadata.get('section', ''), '')}"
                    if doc.metadata.get('section', '') in section_titles else ""
                )
            )
            + f"\n\nFrom: {doc.metadata.get('book', 'Unknown Book')}\n\n"
            + f"{doc.page_content.strip()}"
        )
        for doc in top_docs
    ])

    # Summarize using LLM
    input_data = {"context": formatted_context, "query": query}
    response = (prompt_law | llm | StrOutputParser()).invoke(input_data)

    return response







#Judment

@tool

def judgment_tool(query: str) -> str:
    """
    Use this tool when the user asks about how Pakistani courts have decided similar cases in the past.
    It retrieves and summarizes legal judgments, precedents, or case law from a database of actual court decisions.

    Example queries:
    - "What did the court decide in cheque dishonor cases?"
    - "Any Supreme Court ruling on Section 489-F?"
    - "Has Section 302 ever been challenged in court?"

    The tool returns a summary of top relevant legal judgments (default: 3) based on the user's query.
    """
    k = 3  # Number of top judgments to return

    # Retrieve relevant court judgments
    retrieved = retriever_judgment.invoke(query)

    if not retrieved:
        return "No relevant judgments found."

    # Take top-k documents only
    top_docs = retrieved[:k]

    # Summarize each
    results = []
    for doc in top_docs:
        context = doc.page_content
        input_data = {"context": context}
        response = (prompt_judgment | llm | StrOutputParser()).invoke(input_data)
        results.append(response)

    return "\n\n-------\n\n".join(results)


#Predictor

@tool
def predict_verdict_from_text(text: str) -> str:
    """
    Use this tool when the user provides raw legal case text (such as FIRs, arguments, case summaries, or judgment excerpts)
    and wants to predict what the verdict might be based on the structured information in that text.

    The tool extracts structured fields such as:
    - Petitioner Argument
    - Respondent Argument
    - Offence Sections
    - Case Type
    - Case Summary

    Then it uses a trained LegalBERT-based model to predict the likely court verdict.

    Example queries:
    - "Predict the outcome of this case: [paste full case text]"
    - "Based on this FIR and arguments, what might the court decide?"
    - "Here's a summary and sections applied. Predict the result."

    The tool returns the predicted verdict such as: 'Guilty', 'Not Guilty', 'Dismissed', 'Granted', etc., depending on the case type.
    """
    structured_case = extract_case_chain.invoke({"text": text})
    verdict = predict_legal_verdict(structured_case)
    return verdict





##Agent

In [ ]:
!pip install -U langchain-community -q

In [ ]:
from langchain.agents import initialize_agent, AgentType
from langchain.memory import ConversationBufferMemory
from langchain.chat_models import ChatOpenAI

tools = [
    judgment_tool,
    lawbook_tool,
    predict_verdict_from_text
]

memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)



agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.OPENAI_FUNCTIONS,
    verbose=True,
    memory = memory
)




In [ ]:
response = agent.invoke("What are the legal consequences of cyber harassment under Pakistani law?")
print(response)



> Entering new AgentExecutor chain...

Invoking: `lawbook_tool` with `{'query': 'Cyber harassment in Pakistani law'}`


---

Section 151 – Impeaching credit of witness
From: Qanun-e-Shahadat Order 1984

Section 151 of the Qanun-e-Shahadat Order 1984 pertains to impeaching the credit of a witness. In cases of cyber harassment in Pakistani law, this section may be relevant when questioning the credibility of witnesses providing testimony related to the offense. It allows for the credibility of a witness to be challenged through cross-examination or other means to ensure the accuracy and truthfulness of their statements.

---

Section 188 – Liability of offences committed Outside Pakistan. Political Agents to certify fitness of inquiry into charges.
From: Code Of Criminal Procedure 1898

Section 188 of the Code of Criminal Procedure 1898 deals with the liability of offenses committed outside Pakistan. While cyber harassment may involve elements that extend beyond national borders, this 

##Gradio

In [ ]:
import gradio as gr
from langchain.agents import initialize_agent, AgentType
from langchain.memory import ConversationBufferMemory
from langchain.chat_models import ChatOpenAI
from langchain.schema import StrOutputParser

#LangChain Setup -------------------
llm = ChatOpenAI(temperature=0)


@tool
def lawbook_tool(query: str) -> str:
    """Use this tool for legal provisions or sections (e.g., 489-F, PECA, PPC)."""
    return get_lawbook_sections(query)

@tool
def judgment_tool(query: str) -> str:
    """Use this tool to fetch past case judgments relevant to a legal issue."""
    return get_past_judgments(query)

@tool
def predict_verdict_from_text(raw_text: str) -> str:
    """Use this tool when a full legal case is given for verdict prediction."""
    return predict_verdict(raw_text)

tools = [judgment_tool, lawbook_tool, predict_verdict_from_text]

memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.OPENAI_FUNCTIONS,
    verbose=True,
    memory=memory
)


#Functional Blocks

def predict_verdict(raw_text):
    structured_case = extract_case_chain.invoke({"text": raw_text})
    verdict = predict_legal_verdict(structured_case)
    return verdict

def get_lawbook_sections(query):
    retrieved = retriever_law.invoke(query)
    if not retrieved:
        return "No relevant sections found."

    top_docs = retrieved[:3]
    results = []
    for doc in top_docs:
        context = (
            f"Section {doc.metadata.get('section', 'N/A')} – {doc.metadata.get('title', 'No Title')}\n\n"
            f"From: {doc.metadata.get('book', 'Unknown Book')}\n\n"
            f"{doc.page_content.strip()}"
        )
        prompt_input = {"context": context, "query": query}
        response = (prompt_law | llm | StrOutputParser()).invoke(prompt_input)
        results.append(response)

    return "\n\n---\n\n".join(results)

def get_past_judgments(query):
    retrieved = retriever_judgment.invoke(query)
    if not retrieved:
        return "No relevant judgments found."

    top_docs = retrieved[:3]
    results = []
    for doc in top_docs:
        context = doc.page_content
        response = (prompt_judgment | llm | StrOutputParser()).invoke({"context": context})
        results.append(response)

    return "\n\n---\n\n".join(results)



#Chat Function with Memory -------------------

def chat_fn(message, chat_history):
    try:
        chat_history = chat_history or []

#Build context from chat history
        history_text = ""
        for user_msg, ai_msg in chat_history:
            history_text += f"User: {user_msg}\nAI: {ai_msg}\n"

        full_input = f"{history_text}User: {message}"


#Optional manual routing based on keywords
        if any(word in message.lower() for word in ["section", "law", "pecca", "ppc", "crpc"]):
            reply = lawbook_tool.invoke(message)
        else:
            reply = agent.invoke({"input": full_input})["output"]

        chat_history.append((message, reply))
        return chat_history, chat_history

    except Exception as e:
        error_msg = f"Error: {str(e)}"
        chat_history = chat_history or []
        chat_history.append((message, error_msg))
        return chat_history, chat_history


#Gradio UI

with gr.Blocks(title="LegalMind AI: Justice Meets Intelligence") as demo:
    gr.Markdown("## ⚖️ **LegalMind AI** — Justice Meets Intelligence")


#Predict Verdict Tab
    with gr.Tab("🔮 Predict Verdict"):
        input_text = gr.Textbox(lines=15, label="Enter Legal Case Text")
        verdict_output = gr.Code(label="Predicted Verdict", language="python")
        verdict_btn = gr.Button("Predict Verdict")
        verdict_btn.click(predict_verdict, inputs=input_text, outputs=verdict_output)


# 📚 Law Book Tab
    with gr.Tab("📚 Search Law Books"):
        law_query = gr.Textbox(label="Ask about a legal topic (e.g., cybercrime, 489-F, bail)", lines=3)
        law_output = gr.Code(label="Relevant Legal Sections", language="python")
        law_btn = gr.Button("Search Law Books")
        law_btn.click(get_lawbook_sections, inputs=law_query, outputs=law_output)


# 📜 Past Judgments Tab
    with gr.Tab("📜 Past Case Judgments"):
        case_query = gr.Textbox(label="Describe the legal issue (e.g., 'cheque dishonor under 489-F')", lines=3)
        judgment_output = gr.Code(label="Top Past Judgments", language="python")
        judgment_btn = gr.Button("Find Past Judgments")
        judgment_btn.click(get_past_judgments, inputs=case_query, outputs=judgment_output)


    # 💬 ChatBot Tab
    with gr.Tab("💬 Legal ChatBot"):
        chatbot = gr.Chatbot(label="🧠 LegalMind Chat", height=400)
        msg = gr.Textbox(label="Ask your legal question...")
        state = gr.State([])

        send_btn = gr.Button("Send")
        send_btn.click(chat_fn, inputs=[msg, state], outputs=[chatbot, state])
        msg.submit(chat_fn, inputs=[msg, state], outputs=[chatbot, state])

demo.launch(share=True)




/tmp/ipython-input-53-3699164608.py:143: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(label="🧠 LegalMind Chat", height=400)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://119b36734f4e3478c3.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
